In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# פותר האופטימיזציה: פונקציית Qiskit מאת Q-CTRL Fire Opal
*ראה את [ממשק ה-API](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)*

> **Note:** פונקציות Qiskit הן תכונה ניסיונית הזמינה רק למשתמשי IBM Quantum&reg; Premium Plan, Flex Plan, ו-On-Prem (דרך IBM Quantum Platform API). הן בסטטוס הפצה בתצוגה מקדימה ועשויות להשתנות.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## סקירה כללית
עם Fire Opal Optimization Solver, תוכל לפתור בעיות אופטימיזציה בקנה-מידה שימושי על חומרה קוונטית מבלי שתידרש מומחיות קוונטית. פשוט הזן את הגדרת הבעיה ברמה גבוהה, והפותר ידאג לכל השאר. כל תהליך העבודה מודע לרעש ומנצל את [ניהול הביצועים של Fire Opal](/guides/q-ctrl-performance-management) מאחורי הקלעים. הפותר מספק באופן עקבי פתרונות מדויקים לבעיות מאתגרות קלאסית, אפילו בקנה-מידה מלא של המכשיר על ה-QPU הגדולים ביותר של IBM&reg;.

הפותר גמיש וניתן להשתמש בו לפתרון בעיות אופטימיזציה קומבינטורית המוגדרות כפונקציות מטרה או כגרפים שרירותיים. אין צורך למפות בעיות לטופולוגיה של המכשיר. ניתן לפתור בעיות גם ללא אילוצים וגם עם אילוצים, כל עוד ניתן לנסח את האילוצים כמונחי עונשין. הדוגמאות הכלולות במדריך זה מדגימות כיצד לפתור בעיית אופטימיזציה ללא אילוצים ועם אילוצים בקנה-מידה שימושי, תוך שימוש בסוגי קלט שונים של הפותר. הדוגמה הראשונה כוללת בעיית max-cut המוגדרת על גרף בעל 156 צמתים ו-3-Regular, בעוד הדוגמה השנייה מתמודדת עם בעיית Minimum Vertex Cover בעלת 50 צמתים המוגדרת על ידי פונקציית עלות.

כדי לקבל גישה ל-Optimization Solver, [צור קשר עם Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## תיאור הפונקציה
הפותר מבצע אופטימיזציה מלאה ואוטומציה של כל האלגוריתם, החל מדיכוי שגיאות ברמת החומרה ועד מיפוי בעיות יעיל ואופטימיזציה קלאסית בלולאה סגורה. מאחורי הקלעים, הצינור של הפותר מצמצם שגיאות בכל שלב, ומאפשר את הביצועים המשופרים הנדרשים לסקאלה משמעותית. תהליך העבודה הבסיסי מבוסס על Quantum Approximate Optimization Algorithm (QAOA), שהוא אלגוריתם היברידי קוונטי-קלאסי. לסיכום מפורט של תהליך העבודה המלא של Optimization Solver, ראה ב[מאמר המפורסם](https://arxiv.org/abs/2406.01743).

![ויזואליזציה של תהליך העבודה של Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

כדי לפתור בעיה גנרית עם Optimization Solver:
1. הגדר את הבעיה שלך כפונקציית מטרה, גרף, או שרשרת ספין `SparsePauliOp`.
2. התחבר לפונקציה דרך Qiskit Functions Catalog.
3. הרץ את הבעיה עם הפותר וקבל תוצאות.
### פורמטי בעיה מקובלים
- ייצוג ביטוי פולינומיאלי של פונקציית מטרה. רצוי ליצור בפייתון עם אובייקט SymPy Poly קיים ולעצב לסטרינג באמצעות [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- ייצוג גרף של סוג בעיה ספציפי. יש ליצור את הגרף באמצעות ספריית networkx בפייתון, ולאחר מכן להמיר לסטרינג באמצעות פונקציית networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- ייצוג שרשרת ספין של בעיה ספציפית. שרשרת הספין צריכה להיות מיוצגת כאובייקט `SparsePauliOp`; ראה את [התיעוד](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) לפרטים נוספים.

> **Note:** אם ברצונך להשתמש ב-Backend שפונקציה זו אינה תומכת בו כרגע, [פנה ל-Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) להוספת תמיכה.
## ביצועים השוואתיים
[תוצאות ביצועים השוואתיים מפורסמות](https://arxiv.org/abs/2406.01743) מראות שהפותר פותר בהצלחה בעיות עם יותר מ-120 qubits, ואף עולה על תוצאות שפורסמו בעבר על מכשירי quantum annealing ומלכודות יונים. מדדי הביצועים ההשוואתיים הבאים מספקים אינדיקציה גסה לדיוק וסקאלה של סוגי בעיות בהתבסס על מספר דוגמאות. המדדים בפועל עשויים להיות שונים בהתבסס על תכונות בעיה שונות, כגון מספר המונחים בפונקציית המטרה (צפיפות) והמקומיות שלהם, מספר המשתנים, וסדר הפולינום.

ה"מספר של qubits" המצוין אינו מגבלה קשיחה אלא מייצג סף גס שבו תוכל לצפות לדיוק פתרון עקבי מאוד. גדלי בעיות גדולים יותר נפתרו בהצלחה, וניסוי מעבר למגבלות אלו מומלץ.

קישוריות qubit שרירותית נתמכת בכל סוגי הבעיות.

| סוג בעיה    | מספר qubits | דוגמה | דיוק | זמן כולל (שניות) | שימוש בזמן ריצה (שניות) | מספר איטרציות
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| בעיות ריבועיות עם קישוריות דלילה  | 156 | 3-regular max-cut| 100%     | 1764     | 293          | 16 |
| אופטימיזציה בינארית מסדר גבוה | 156 | מודל Ising spin-glass | 100%      | 1461     | 272          | 16 |
| בעיות ריבועיות עם קישוריות צפופה | 50 | Fully-connected max-cut| 100%      |  1758    | 268  | 12 |
| בעיה עם אילוצים ומונחי עונשין | 50 | Weighted Minimum Vertex Cover עם צפיפות קשתות 8% | 100%      | 1074     | 215 | 10 |
## תחילת העבודה
ראשית, אמת את עצמך באמצעות [מפתח ה-API של IBM Quantum](http://quantum.cloud.ibm.com/). לאחר מכן, בחר את פונקציית Qiskit כדלקמן. (קטע זה מניח שכבר [שמרת את חשבונך](/guides/functions#install-qiskit-functions-catalog-client) בסביבה המקומית שלך.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

### 1. הגדר את הבעיה
תוכל להריץ בעיית Max-Cut על ידי הגדרת בעיית גרף וציון `problem_type='maxcut'`.

In [3]:
# %pip install networkx numpy

### 1. Define the problem
You can run a max-cut problem by defining a graph problem and specifying `problem_type='maxcut'`.

In [5]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [6]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

### 2. הרץ את הבעיה
בעת שימוש בשיטת קלט מבוססת גרף, ציין את סוג הבעיה.

In [7]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Run the problem
When using the graph-based input method, specify the problem type.

In [8]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [9]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [10]:
# Print the ID so you can use it later, if necessary
print(maxcut_job.job_id)

# Get job status
print(maxcut_job.status())

ba5fbdb8-9e71-49bd-8320-79dcdb46de6a
QUEUED


### 3. קבל את התוצאה
קבל את ערך החיתוך האופטימלי ממילון התוצאות.

> **Note:** המיפוי של המשתנים ל-bitstring עשוי להשתנות. מילון הפלט מכיל תת-מילון `variables_to_bitstring_index_map`, אשר עוזר לאמת את הסדר.

In [11]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior max-cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [12]:
# %pip install numpy networkx sympy

תוכל לאמת את דיוק התוצאה על ידי פתרון הבעיה קלאסית עם פותרים בקוד פתוח כמו [PuLP](https://coin-or.github.io/pulp/) אם הגרף אינו מחובר בצפיפות. בעיות עם צפיפות גבוהה עשויות לדרוש פותרים קלאסיים מתקדמים כדי לאמת את הפתרון.
## דוגמה: אופטימיזציה עם אילוצים
דוגמת max-cut הקודמת היא בעיית אופטימיזציה בינארית ריבועית ללא אילוצים נפוצה. Optimization Solver של Q-CTRL יכול לשמש לסוגי בעיות שונים, כולל אופטימיזציה עם אילוצים. תוכל לפתור סוגי בעיות שרירותיים על ידי הזנת הגדרת הבעיה המיוצגת כפולינום שבו אילוצים מדוגמים כמונחי עונשין.

הדוגמה הבאה מדגימה כיצד לבנות פונקציית עלות עבור בעיית אופטימיזציה עם אילוצים, [כיסוי קודקוד מינימלי](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
בנוסף לחבילות `qiskit-ibm-catalog` ו-`qiskit`, תשתמש גם בחבילות הבאות להרצת דוגמה זו: `numpy`, `networkx`, ו-`sympy`. תוכל להתקין חבילות אלו על ידי הסרת הסימון מהתא הבא אם אתה מריץ דוגמה זו ב-notebook באמצעות ה-kernel של IPython.

In [13]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

### 1. הגדר את הבעיה
הגדר בעיית MVC אקראית על ידי יצירת גרף עם צמתים בעלי משקל אקראי.

In [14]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

![פלט תא הקוד הקודם](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

ניתן לנסח מודל אופטימיזציה סטנדרטי ל-MVC ממושקל כדלקמן. ראשית, יש להוסיף עונש עבור כל מקרה שבו קשת אינה מחוברת לקודקוד בתת-הקבוצה. לכן, נגדיר $n_i = 1$ אם קודקוד $i$ נמצא בכיסוי (כלומר, בתת-הקבוצה) ו-$n_i = 0$ אחרת. שנית, המטרה היא למזער את המספר הכולל של הקודקודים בתת-הקבוצה, אשר ניתן לייצג באמצעות הפונקציה הבאה:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [15]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

כעת כל קשת בגרף צריכה לכלול לפחות נקודת קצה אחת מהכיסוי, אשר ניתן לבטא כאי-שוויון:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

כל מקרה שבו קשת אינה מחוברת לקודקוד הכיסוי חייב לקבל עונש. ניתן לייצג זאת בפונקציית העלות על ידי הוספת עונש בצורה $P(1-n_i-n_j+n_i n_j)$ כאשר $P$ הוא קבוע עונשין חיובי. לפיכך, חלופה ללא אילוצים לאי-השוויון עם אילוצים ל-MVC ממושקל היא:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [16]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 2. הרץ את הבעיה

In [17]:
print(mvc_job.status())

QUEUED


בדוק את [הסטטוס](/guides/functions#check-job-status) של עומס העבודה של פונקציית Qiskit שלך או קבל [תוצאות](/guides/functions#retrieve-results) כדלקמן:

In [18]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


## Get support

For any questions or issues, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com).

## Changelog

- 2026-02-11: We now have support for `ibm_miami`

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Q-CTRL Optimization Solver](https://quantum.cloud.ibm.com/functions?id=q-ctrl-optimization-solver).
- Visit the [API reference](/docs/api/functions/q-ctrl-optimization-solver) for this Qiskit Function.
- Try the [Solve higher-order binary optimization problems with Q-CTRL's Optimization Solver](/docs/tutorials/solve-higher-order-binary-optimization-problems-with-q-ctrls-optimization-solver) tutorial.
- Review [Sachdeva, N., et al. (2024).  Quantum optimization using a 127-qubit gate-model IBM quantum computer can outperform quantum annealers for nontrivial binary optimization problems. arXiv preprint arXiv:2406.01743](https://arxiv.org/abs/2406.01743).
- Review [Loco, D., et al. (2026).  Practical protein-pocket hydration-site prediction for drug discovery on a quantum computer. arXiv preprint arXiv:2512.08390](https://arxiv.org/abs/2512.08390).
- Review the [Mazda](https://q-ctrl.com/case-study/tackling-a-costly-bottleneck-in-automotive-design) case study.
- Review the [Network Rail](https://q-ctrl.com/case-study/accelerating-the-schedule-for-quantum-enhanced-rail) case study.
- Review the [Australian Army](https://q-ctrl.com/case-study/improving-army-logistics-with-quantum-computing) case study.
- Review the [Transport for New South Wales](https://q-ctrl.com/case-study/delivering-quantum-computing-for-faster-commuting) case study.

</Admonition>